# Exploring `mujoco/invertedpendulum/expert-v0` with Minari

This notebook loads the Minari expert dataset for the MuJoCo InvertedPendulum environment,
assembles a **100 000 × 5** observation-action matrix, and lays the groundwork for training
a Self-Organizing Map (SOM) on the resulting data.

| Column index | Description |
|---|---|
| 0 | Cart position |
| 1 | Cart velocity |
| 2 | Pole angle |
| 3 | Pole angular velocity |
| 4 | Action (force applied to cart) |

## 1  Imports

In [ ]:
import numpy as np
import minari
from minisom import MiniSom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## 2  Load dataset

If the dataset has not been downloaded yet, Minari will fetch it automatically from the remote
registry the first time `load_dataset` is called.

In [ ]:
DATASET_ID = "mujoco/invertedpendulum/expert-v0"

# Download if not already cached locally
if DATASET_ID not in minari.list_local_datasets():
    minari.download_dataset(DATASET_ID)

dataset = minari.load_dataset(DATASET_ID)
print(f"Dataset loaded: {DATASET_ID}")
print(f"Total episodes : {dataset.total_episodes}")
print(f"Total steps    : {dataset.total_steps}")

## 3  Inspect one episode

Each episode is an `EpisodeData` object with numpy arrays for `observations`, `actions`,
`rewards`, `terminations`, and `truncations`.

> **Note**: Minari stores one *extra* observation per episode (the terminal observation is
> appended), so `observations` has shape `(T+1, obs_dim)` while `actions` has shape
> `(T, act_dim)`.  We use only the first `T` observations to form aligned pairs.

In [ ]:
sample_episode = next(iter(dataset.iterate_episodes()))

print("observations shape :", sample_episode.observations.shape)
print("actions shape      :", sample_episode.actions.shape)
print("rewards shape      :", sample_episode.rewards.shape)

## 4  Build the 100 000 × 5 observation-action matrix

In [ ]:
TARGET_ROWS = 100_000

obs_list = []
act_list = []

for episode in dataset.iterate_episodes():
    obs = episode.observations  # shape (T+1, 4)
    act = episode.actions       # shape (T, 1)
    T = act.shape[0]
    obs_list.append(obs[:T])    # align: drop terminal observation
    act_list.append(act)

    collected = sum(a.shape[0] for a in act_list)
    if collected >= TARGET_ROWS:
        break

all_obs = np.concatenate(obs_list, axis=0)   # (N, 4)
all_act = np.concatenate(act_list, axis=0)   # (N, 1)

# Trim or pad to exactly TARGET_ROWS
all_obs = all_obs[:TARGET_ROWS]
all_act = all_act[:TARGET_ROWS]

# Combine into a single (100_000, 5) matrix
data_matrix = np.hstack([all_obs, all_act])  # (100_000, 5)

print("data_matrix shape:", data_matrix.shape)
assert data_matrix.shape == (TARGET_ROWS, 5), (
    f"Expected ({TARGET_ROWS}, 5) but got {data_matrix.shape}"
)

## 5  Summary statistics

In [ ]:
COL_NAMES = ["cart_pos", "cart_vel", "pole_angle", "pole_ang_vel", "action"]

print(f"{'Column':<18} {'Min':>10} {'Max':>10} {'Mean':>10} {'Std':>10}")
print("-" * 62)
for i, name in enumerate(COL_NAMES):
    col = data_matrix[:, i]
    print(f"{name:<18} {col.min():>10.4f} {col.max():>10.4f} "
          f"{col.mean():>10.4f} {col.std():>10.4f}")

## 6  Normalise with scikit-learn `StandardScaler`

In [ ]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_matrix)  # (100_000, 5), zero-mean unit-variance

print("Scaled data — mean per column :", data_scaled.mean(axis=0).round(6))
print("Scaled data — std  per column :", data_scaled.std(axis=0).round(6))

## 7  Quick PCA sanity check (scikit-learn)

In [ ]:
pca = PCA(n_components=5)
pca.fit(data_scaled)

print("Explained variance ratio per component:")
for i, ratio in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ratio:.4f}  (cumulative: {pca.explained_variance_ratio_[:i+1].sum():.4f})")

## 8  Train a Self-Organizing Map (MiniSom)

A small 10 × 10 SOM is trained on the scaled data as a starting-point for exploration.
Adjust `som_x`, `som_y`, `sigma`, `learning_rate`, and `num_iteration` as needed.

In [ ]:
SOM_X = 10
SOM_Y = 10
INPUT_DIM = data_scaled.shape[1]  # 5

som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=INPUT_DIM,
    sigma=1.5,
    learning_rate=0.5,
    random_seed=42,
)

som.random_weights_init(data_scaled)
som.train(data_scaled, num_iteration=10_000, verbose=True)

print("\nSOM training complete.")
print(f"Weight matrix shape: {som.get_weights().shape}")

## 9  Quantisation error

The quantisation error is the mean distance between each data point and its Best Matching
Unit (BMU) — a lower value indicates a better-fitting map.

In [ ]:
q_error = som.quantization_error(data_scaled)
print(f"Quantisation error: {q_error:.6f}")